In [1]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

DATA_DIR = Path("/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset")

print("Loading businesses...")
businesses = []
with open(DATA_DIR / "yelp_academic_dataset_business.json") as f:
    for line in f:
        b = json.loads(line)
        businesses.append(b)

biz_df = pd.DataFrame(businesses)[["business_id", "name", "state", "city", "categories"]]
print(biz_df["state"].value_counts().head(15))
print(f"Total businesses: {len(biz_df)}")

Loading businesses...
state
PA    34039
FL    26330
TN    12056
IN    11247
MO    10913
LA     9924
AZ     9912
NJ     8536
NV     7715
AB     5573
CA     5203
ID     4467
DE     2265
IL     2145
TX        4
Name: count, dtype: int64
Total businesses: 150346


review = engagement (like/retweet)

tip = low-coverage ad engagement

friend = follow

In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

DATA_DIR = Path("/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset")
OUT_DIR = Path("/kaggle/working/yelp_graph")
OUT_DIR.mkdir(exist_ok=True)

STATE = "PA"

print("Loading businesses")
businesses = []
with open(DATA_DIR / "yelp_academic_dataset_business.json") as f:
    for line in f:
        b = json.loads(line)
        businesses.append(b)

biz_df = pd.DataFrame(businesses)[["business_id", "name", "state", "city", "categories"]]
biz_pa = biz_df[biz_df["state"] == STATE].copy()
biz_ids = set(biz_pa["business_id"])
print(f"Businesses in {STATE}: {len(biz_pa)}")

print("Loading reviews")
reviews = []
with open(DATA_DIR / "yelp_academic_dataset_review.json") as f:
    for line in f:
        r = json.loads(line)
        if r["business_id"] in biz_ids:
            reviews.append({
                "user_id": r["user_id"],
                "business_id": r["business_id"],
                "date": r["date"],
                "stars": r["stars"],
                "useful": r["useful"]
            })
review_df = pd.DataFrame(reviews)
print(f"Reviews: {len(review_df)}, unique users: {review_df['user_id'].nunique()}")

print("Loading tips")
tips = []
with open(DATA_DIR / "yelp_academic_dataset_tip.json") as f:
    for line in f:
        t = json.loads(line)
        if t["business_id"] in biz_ids:
            tips.append({
                "user_id": t["user_id"],
                "business_id": t["business_id"],
                "date": t["date"]
            })
tip_df = pd.DataFrame(tips)
print(f"Tips: {len(tip_df)}, unique users: {tip_df['user_id'].nunique()}")

print("Loading checkins")
checkins = []
with open(DATA_DIR / "yelp_academic_dataset_checkin.json") as f:
    for line in f:
        c = json.loads(line)
        if c["business_id"] in biz_ids:
            for date in c["date"].split(", "):
                checkins.append({
                    "business_id": c["business_id"],
                    "date": date.strip()
                })
checkin_df = pd.DataFrame(checkins)
print(f"Checkin events: {len(checkin_df)}")

Loading businesses
Businesses in PA: 34039
Loading reviews
Reviews: 1598960, unique users: 430688
Loading tips
Tips: 193609, unique users: 65182
Loading checkins
Checkin events: 2710714


In [3]:
pa_user_ids = set(review_df["user_id"]) | set(tip_df["user_id"])
print(f"PA users (from interactions): {len(pa_user_ids)}")

print("Loading users + friends")
users = []
friend_edges = []

with open(DATA_DIR / "yelp_academic_dataset_user.json") as f:
    for line in f:
        u = json.loads(line)
        if u["user_id"] not in pa_user_ids:
            continue
        users.append({
            "user_id": u["user_id"],
            "review_count": u["review_count"],
            "fans": u["fans"],
            "average_stars": u["average_stars"]
        })
        # friends - если оба конца есть в PA
        if u["friends"] and u["friends"] != "None":
            for friend_id in u["friends"].split(", "):
                friend_id = friend_id.strip()
                if friend_id in pa_user_ids:
                    friend_edges.append({
                        "src": u["user_id"],
                        "dst": friend_id
                    })

user_df = pd.DataFrame(users)
friends_df = pd.DataFrame(friend_edges) if friend_edges else pd.DataFrame(columns=["src","dst"])

# убираем AB или BA
if len(friends_df) > 0:
    friends_df["key"] = friends_df.apply(lambda r: tuple(sorted([r.src, r.dst])), axis=1)
    friends_df = friends_df.drop_duplicates("key").drop(columns="key").reset_index(drop=True)

print(f"Users loaded: {len(user_df)}")
print(f"Friend edges (deduplicated): {len(friends_df)}")

PA users (from interactions): 434918
Loading users + friends
Users loaded: 434914
Friend edges (deduplicated): 1271665


In [4]:
all_users = list(set(review_df["user_id"]) | set(tip_df["user_id"]))
all_businesses = list(biz_pa["business_id"])

user2id = {u: i for i, u in enumerate(all_users)}
biz2id  = {b: i for i, b in enumerate(all_businesses)}

print(f"Entity counts:")
print(f"Users:      {len(user2id)}")
print(f"Businesses: {len(biz2id)}")

pd.DataFrame({"user_id": list(user2id.keys()), "node_id": list(user2id.values())}) \
    .to_csv(OUT_DIR / "user2id.csv", index=False)
pd.DataFrame({"business_id": list(biz2id.keys()), "node_id": list(biz2id.values())}) \
    .to_csv(OUT_DIR / "biz2id.csv", index=False)

print("Mappings saved")

Entity counts:
Users:      434918
Businesses: 34039
Mappings saved


In [5]:
# review: user - business (high-coverage engagement)
review_edges = pd.DataFrame({
    "src": review_df["user_id"].map(user2id),
    "dst": review_df["business_id"].map(biz2id),
    "relation": "review"
}).dropna().astype({"src": int, "dst": int})

# tip: user - business (low-coverage engagement)
tip_edges = pd.DataFrame({
    "src": tip_df["user_id"].map(user2id),
    "dst": tip_df["business_id"].map(biz2id),
    "relation": "tip"
}).dropna().astype({"src": int, "dst": int})

# friends: user - user (social graph)
if len(friends_df) > 0:
    friend_edges_mapped = pd.DataFrame({
        "src": friends_df["src"].map(user2id),
        "dst": friends_df["dst"].map(user2id),
        "relation": "friend"
    }).dropna().astype({"src": int, "dst": int})
else:
    friend_edges_mapped = pd.DataFrame(columns=["src","dst","relation"])

print("Edge counts per relation:")
print(f"review:  {len(review_edges)}")
print(f"tip:     {len(tip_edges)}")
print(f"friend:  {len(friend_edges_mapped)}")
print(f"total:   {len(review_edges) + len(tip_edges) + len(friend_edges_mapped)}")

Edge counts per relation:
review:  1598960
tip:     193609
friend:  1271665
total:   3064234


In [6]:
from sklearn.model_selection import train_test_split

def split_and_save(edges_df, relation_name, out_dir, val_ratio=0.05, test_ratio=0.1):
    idx = np.arange(len(edges_df))
    train_idx, temp_idx = train_test_split(idx, test_size=val_ratio+test_ratio, random_state=42)
    val_idx, test_idx   = train_test_split(temp_idx, test_size=test_ratio/(val_ratio+test_ratio), random_state=42)

    for split_name, split_idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        split_df = edges_df.iloc[split_idx][["src", "dst"]]
        path = out_dir / f"{relation_name}_{split_name}.tsv"
        split_df.to_csv(path, sep="\t", index=False, header=False)

    print(f"{relation_name}: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

print("Saving edge splits")
split_and_save(review_edges,"review", OUT_DIR)
split_and_save(tip_edges, "tip", OUT_DIR)
if len(friend_edges_mapped) > 0:
    split_and_save(friend_edges_mapped, "friend", OUT_DIR)

print("\nFiles in output dir:")
for f in sorted(OUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"{f.name:40s} {size_mb:.1f} MB")

Saving edge splits
review: train=1359115, val=79948, test=159897
tip: train=164567, val=9680, test=19362
friend: train=1080915, val=63583, test=127167

Files in output dir:
biz2id.csv                               1.0 MB
friend_test.tsv                          1.7 MB
friend_train.tsv                         14.6 MB
friend_val.tsv                           0.9 MB
review_test.tsv                          2.0 MB
review_train.tsv                         16.9 MB
review_val.tsv                           1.0 MB
tip_test.tsv                             0.2 MB
tip_train.tsv                            2.0 MB
tip_val.tsv                              0.1 MB
user2id.csv                              12.9 MB


In [7]:
print("Graph statistics")
print(f"State filter: {STATE}")
print("Nodes:")
print(f"Users: {len(user2id):>8,}")
print(f"Businesses: {len(biz2id):>8,}")
print(f"Total: {len(user2id)+len(biz2id):>8,}")
print()
print("Edges (before split):")
print(f"review: {len(review_edges):>8,}  (user-business, high-coverage)")
print(f"tip: {len(tip_edges):>8,}  (user-business, low-coverage)")
print(f"friend: {len(friend_edges_mapped):>8,}  (user-user, social)")
total = len(review_edges) + len(tip_edges) + len(friend_edges_mapped)
print(f"Total: {total:>8,}")

Graph statistics
State filter: PA
Nodes:
Users:  434,918
Businesses:   34,039
Total:  468,957

Edges (before split):
review: 1,598,960  (user-business, high-coverage)
tip:  193,609  (user-business, low-coverage)
friend: 1,271,665  (user-user, social)
Total: 3,064,234


In [11]:
print("Частота связей (low/high coverage)")

total_users = len(user2id)
review_users = review_edges["src"].nunique()
tip_users = tip_edges["src"].nunique()
friend_users = friend_edges_mapped["src"].nunique() if len(friend_edges_mapped) > 0 else 0

print(f"review: {len(review_edges):>8,} рёбер | {review_users:>7,} уникальных пользователей ({100*review_users/total_users:.1f}%)")
print(f"tip: {len(tip_edges):>8,} рёбер | {tip_users:>7,} уникальных пользователей ({100*tip_users/total_users:.1f}%)")
print(f"friend: {len(friend_edges_mapped):>8,} рёбер | {friend_users:>7,} уникальных пользователей ({100*friend_users/total_users:.1f}%)")

print("\nСреднее рёбер на пользователя")
print(f"review: {len(review_edges)/total_users:.2f}")
print(f"tip: {len(tip_edges)/total_users:.2f}")
print(f"friend: {len(friend_edges_mapped)/total_users:.2f}")

Частота связей (low/high coverage)
review: 1,598,960 рёбер | 430,688 уникальных пользователей (99.0%)
tip:  193,609 рёбер |  65,182 уникальных пользователей (15.0%)
friend: 1,271,665 рёбер | 126,300 уникальных пользователей (29.0%)

Среднее рёбер на пользователя
review: 3.68
tip: 0.45
friend: 2.92


In [9]:
ablations = {
    "all": [review_edges, tip_edges, friend_edges_mapped],
    "no_tip": [review_edges, friend_edges_mapped],
    "no_friend": [review_edges, tip_edges],
    "review_only": [review_edges],
    "tip_only": [tip_edges],
    "friend_only": [friend_edges_mapped],
}

for name, edge_list in ablations.items():
    abl_dir = OUT_DIR / f"ablation_{name}"
    abl_dir.mkdir(exist_ok=True)
    
    all_edges = pd.concat(edge_list).reset_index(drop=True) if edge_list else pd.DataFrame()
    
    if len(all_edges) == 0:
        continue
    
    # train/val/test split
    idx = np.arange(len(all_edges))
    train_idx, temp_idx = train_test_split(idx, test_size=0.15, random_state=42)
    val_idx, test_idx   = train_test_split(temp_idx, test_size=0.667, random_state=42)
    
    for split_name, split_idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        all_edges.iloc[split_idx][["src", "dst", "relation"]].to_csv(
            abl_dir / f"{split_name}.tsv", sep="\t", index=False, header=False
        )
    
    print(f"ablation '{name}': {len(all_edges):,} рёбер → {abl_dir}")

print("\nПапки ablation_* для обучения - ок")

ablation 'all': 3,064,234 рёбер → /kaggle/working/yelp_graph/ablation_all
ablation 'no_tip': 2,870,625 рёбер → /kaggle/working/yelp_graph/ablation_no_tip
ablation 'no_friend': 1,792,569 рёбер → /kaggle/working/yelp_graph/ablation_no_friend
ablation 'review_only': 1,598,960 рёбер → /kaggle/working/yelp_graph/ablation_review_only
ablation 'tip_only': 193,609 рёбер → /kaggle/working/yelp_graph/ablation_tip_only
ablation 'friend_only': 1,271,665 рёбер → /kaggle/working/yelp_graph/ablation_friend_only

Папки ablation_* для обучения - ок


In [10]:
import shutil
import os

shutil.make_archive('/kaggle/working/output', 'zip', '/kaggle/working')

'/kaggle/working/output.zip'